In [1]:
#!pip install peft
!pip install bert-score -q
!pip install rouge-score -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
#!pip install transformers torch peft

In [3]:
import json
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # For CUDA error debugging

In [4]:
# Step 1: Load the fine-tuned GPT-2 model with LoRA
# model_name = "Salm00n/gpt2_SATACT_v1"

# # Load the tokenizer
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# tokenizer.pad_token = tokenizer.eos_token

# # Load the base GPT-2 model and apply the fine-tuned LoRA adapter
# base_model = AutoModelForCausalLM.from_pretrained("gpt2")  # Base model before fine-tuning
# model = PeftModel.from_pretrained(base_model, model_name, adapter_name="default")

# # Move to GPU if available
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)
# print(f"Loaded fine-tuned model: {model_name} on {device}")

#Step 1: Load GPT-2 XL
model_name = "gpt2-xl"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Loaded fine-tuned model: {model_name} on {device}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded fine-tuned model: gpt2-xl on cuda


# SATACT

# One Shot

In [5]:
#adding [END]
torch.manual_seed(42)
def generate_cot_response_test3(prompt, max_new_tokens=150):  
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
    stop_token_ids = tokenizer.encode("[END]", add_special_tokens=False)
    # Explicitly pass attention_mask to avoid warning
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True, #False is not giving the answer; just produces some text relevant not very meaningful even with increased max_new_tokens
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=stop_token_ids   
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response


# Inline CoT prompt with one-shot example in your style
cot_prompt = (
    "[Example 1]\n"
    "[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist "
    "Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, "
    "OSIRIS-REx successfully _______ a sample of the surface, gathering pieces of it to bring back to Earth.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:"
    "A) attached"
    "B) collected"
    "C) followed"
    "D) replaced"
    "Step-by-step reasoning:\n"
    "Choice B is the best answer because it most logically completes the text’s discussion of the OSIRIS-REx spacecraft’s "
    "contact with the asteroid 101955 Bennu. In this context, “collected” means acquired and took away. The text indicates "
    "that although the boulders on the asteroid’s surface caused some unforeseen problems, OSIRIS-REx was able to gather a "
    "sample to return to Earth. This context suggests that OSIRIS-REx successfully collected a sample of 101955 Bennu.[END]\n\n"
    
    "[Current Problem]\n"
    "[Context]: Research conducted by planetary scientist Katarina Miljkovic suggests that the Moon’s surface may "
    "not accurately _______ early impact events. When the Moon was still forming, its surface was softer, and asteroid or "
    "meteoroid impacts would have left less of an impression; thus, evidence of early impacts may no longer be present.\n"
    "[Question]: Which choice completes the text with the most logical and precise word or phrase?\n"
    "[Options]:\n"
    "A) reflect\n"
    "B) receive\n"
    "C) evaluate\n"
    "D) mimic\n"
    "Step-by-step reasoning:\n"

) 

# Test it
#print(cot_prompt)

# Diagnostic (optional)
inputs = tokenizer(cot_prompt, return_tensors="pt", padding=True)
print(f"Input length: {inputs['input_ids'].shape[1]} tokens")


# Run and display
print("Generating reasoning...\n")
response = generate_cot_response_test3(cot_prompt)
#print("Full Response:\n", response)

#working - print sboth example and current problem with one reasoning: THIS PRINTS THE ENTIRE RESPONSE
print("RESPONSE:\n", response)

Input length: 364 tokens
Generating reasoning...

RESPONSE:
 [Example 1]
[Context]: The spacecraft OSIRIS-REx briefly made contact with the asteroid 101955 Bennu in 2020. NASA scientist Daniella DellaGiustina reports that despite facing the unexpected obstacle of a surface mostly covered in boulders, OSIRIS-REx successfully _______ a sample of the surface, gathering pieces of it to bring back to Earth.
[Question]: Which choice completes the text with the most logical and precise word or phrase?
[Options]:A) attachedB) collectedC) followedD) replacedStep-by-step reasoning:
Choice B is the best answer because it most logically completes the text’s discussion of the OSIRIS-REx spacecraft’s contact with the asteroid 101955 Bennu. In this context, “collected” means acquired and took away. The text indicates that although the boulders on the asteroid’s surface caused some unforeseen problems, OSIRIS-REx was able to gather a sample to return to Earth. This context suggests that OSIRIS-REx suc

# Evaluate

# Bert score and rouge score

In [7]:
from bert_score import score
from rouge_score import rouge_scorer


generated_cot = ("Choice A reflects the text's discussion of how the Moon's surface may have been affected by early impacts. The context indicates that the softness of the Moon may have made it more difficult for early impacts to leave an impression on its surface.’reflect› means to reflect or reflect back. In the context of the text, the Moon reflects the impact of asteroids and meteoroids onto its surface, which may have caused it to be softer than it is today.’receive› is a synonym for ’receiving›, which means to receive or receive back. The Moon receives asteroids and meteors that strike it, which causes it to have a softer surface than it does today. ’")
reference_cot = ("Choice A is the best answer because it most logically completes the text’s discussion of the Moon’s surface. In this context, “reflect” means show or make apparent. The text states that because the surface of the Moon was softer when the Moon was still forming than it is now, early asteroid and meteoroid impacts “would have left less of an impression” and, as a result, evidence of them may no longer exist. This context supports the idea that the surface of the Moon may not accurately show signs of early impact events.")

P, R, F1 = score([generated_cot], [reference_cot], lang="en", model_type="bert-base-uncased")
print(f"BERTScore F1: {F1.item():.3f}")

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
scores = scorer.score(reference_cot, generated_cot)
print(f"ROUGE-1: {scores['rouge1'].fmeasure:.3f}, ROUGE-L: {scores['rougeL'].fmeasure:.3f}")

BERTScore F1: 0.666
ROUGE-1: 0.533, ROUGE-L: 0.267


In [11]:
# # Load NLI model pipeline
# from transformers import pipeline
# nli = pipeline("text-classification", model="facebook/bart-large-mnli")

# def check_entailment(passage, hypothesis):
#     result = nli([passage, hypothesis])[0]
#     return result['label'], round(result['score'], 4)
#     # result = nli({"premise": passage, "hypothesis": hypothesis})[0]
#     # return result['label'], round(result['score'], 4)

# passage  = "Research conducted by planetary scientist Katarina Miljkovic suggests that the Moon’s surface may not accurately _______ early impact events. When the Moon was still forming, its surface was softer, and asteroid or meteoroid impacts would have left less of an impression; thus, evidence of early impacts may no longer be present."
# gpt_base_output = "Choice A is the most correct choice because it is the only one that is consistent with the text of the text. It is also the only choice that does not contradict the text, because it does not follow the text in any way. Example 1: The Moon's surface is covered with boulders and rocks. The moon's surface was covered with rocks and boulders in the early stages of the Moon's orbit, but it was not covered with any rocks or boulders during the early phases of the moon's orbit. The lunar surface is not covered by any rocks, rocks, boulders or other debris. The Moon is covered by a thin layer of dust and dust particles, which may have been deposited on the lunar surface during the"
# gpt_finetuned_output = "Choice A is the best answer because it most logically completes the text’s discussion of the Moon’s surface. In this context, “reflect” means show or make apparent. The text states that because the surface of the Moon was softer when the Moon was still forming than it is now, early asteroid and meteoroid impacts “would have left less of an impression” and, as a result, evidence of them may no longer exist. This context supports the idea that the surface of the Moon may not accurately show signs of early impact events."


# print("=== NLI Entailment from Passage ===")
# base_label, base_score = check_entailment(passage, gpt_base_output)
# fine_label, fine_score = check_entailment(passage, gpt_finetuned_output)

# print(f"Base Model: {base_label} (confidence: {base_score})")
# print(f"Fine-tuned Model: {fine_label} (confidence: {fine_score})")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0


=== NLI Entailment from Passage ===
Base Model: neutral (confidence: 0.9813)
Fine-tuned Model: neutral (confidence: 0.9813)


# Question Type: Replace underlined text with...

In [8]:
#adding [END]
torch.manual_seed(42)
def generate_cot_response_test3(prompt, max_new_tokens=150):  
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
    stop_token_ids = tokenizer.encode("[END]", add_special_tokens=False)
    # Explicitly pass attention_mask to avoid warning
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True, #False is not giving the answer; just produces some text relevant not very meaningful even with increased max_new_tokens
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=stop_token_ids   
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response


# Inline CoT prompt with one-shot example
cot_prompt = (
    "[Example 1]\n"
    "[Context]:The following text is adapted from Cynthia Kadohata's 2004 novel Kira-Kira. [Uncle Katsuhisa] was as loud as my father was quiet. Even when he wasn't talking, he made a lot of noise, clearing his throat and sniffing and tapping his fingers.\n"
    "[Question]:Which choice best describes the function of this '[Uncle Katsuhisa] was as loud as my father was quiet. ' portion of the sentence?\n"
    "[Options]:\n"
    "A) It lists the kinds of topics Uncle Katsuhisa enjoys discussing."
    "B) It suggests that Uncle Katsuhisa dislikes meeting new people."	
    "C) It contrasts Uncle Katsuhisa with the narrator's father."
    "D) It describes a conversation between the narrator and the narrator's father."
    "Step-by-step reasoning:\n"
    "Choice C is the best answer because it most accurately describes how the underlined sentence functions in the text as a whole. The underlined sentence establishes a difference between Uncle Katsuhisa and the narrator’s father by describing Uncle Katsuhisa as “loud” and the narrator’s father as “quiet.” The text then elaborates on that contrast, describing some ways Uncle Katsuhisa is very noisy even when he isn’t speaking.[END]\n\n" 

    "[Current Problem]\n:"
    "[Context]:The following text is from Charlotte Forten Grimké’s 1888 poem “At Newport.” Oh, deep delight to watch the gladsome waves Exultant leap upon the rugged rocks; Ever repulsed, yet ever rushing on—Filled with a life that will not know defeat; To see the glorious hues of sky and sea. The distant snowy sails, glide spirit like, Into an unknown world, to feel the sweet Enchantment of the sea thrill all the soul, Clearing the clouded brain, making the heart Leap joyous as it own bright, singing waves!\n"
    "[Question]:Which choice best describes the function of the portion, “Ever repulsed, yet ever rushing on—Filled with a life that will not know defeat;” in the text as a whole?\n"
    "[Options]:\n"
    "A) It portrays the surroundings as an imposing and intimidating scene.\n"	
    "B) It characterizes the sea’s waves as a relentless and enduring force.\n"
    "C) It conveys the speaker’s ambivalence about the natural world.\n"	
    "D) It draws a contrast between the sea’s waves and the speaker’s thoughts.\n"
    "\nStep-by-step reasoning:\n" 
)

# Test it
#print(cot_prompt)

# Diagnostic (optional)
inputs = tokenizer(cot_prompt, return_tensors="pt", padding=True)
print(f"Input length: {inputs['input_ids'].shape[1]} tokens")


# Run and display
print("Generating reasoning...\n")
response = generate_cot_response_test3(cot_prompt)
#print("Full Response:\n", response)

#working - print sboth example and current problem with one reasoning: THIS PRINTS THE ENTIRE RESPONSE
print("RESPONSE:\n", response)

Input length: 541 tokens
Generating reasoning...

RESPONSE:
 [Example 1]
[Context]:The following text is adapted from Cynthia Kadohata's 2004 novel Kira-Kira. [Uncle Katsuhisa] was as loud as my father was quiet. Even when he wasn't talking, he made a lot of noise, clearing his throat and sniffing and tapping his fingers.
[Question]:Which choice best describes the function of this '[Uncle Katsuhisa] was as loud as my father was quiet. ' portion of the sentence?
[Options]:
A) It lists the kinds of topics Uncle Katsuhisa enjoys discussing.B) It suggests that Uncle Katsuhisa dislikes meeting new people.C) It contrasts Uncle Katsuhisa with the narrator's father.D) It describes a conversation between the narrator and the narrator's father.Step-by-step reasoning:
Choice C is the best answer because it most accurately describes how the underlined sentence functions in the text as a whole. The underlined sentence establishes a difference between Uncle Katsuhisa and the narrator’s father by des

In [9]:
from bert_score import score
from rouge_score import rouge_scorer


generated_cot = ("Choice A is the most accurate description of the text because it conveys how the text functions in a larger context. The text describes the speaker's feelings about the sea as a force that will never be defeated. The speaker is ambivalent about the nature of the world and the sea. He is conflicted about whether the sea is a force of nature or a force created by human beings. The sea is both a natural force and a creation of human beings, but the speaker is unsure about which one is more important. The narrator is similarly ambivalent. He feels that the ocean is both natural and created by humans, but he also feels that it is unnatural and unnatural by human standards. The ocean is neither natural nor unnatural, but it is both")
reference_cot = ("Choice B is the best answer because it most accurately describes how the underlined portion functions in the text as a whole. The text presents the speaker’s experience of viewing the sea. In the underlined portion, the speaker focuses on the idea that the waves hitting rocks on the shore are a relentless and enduring force: they are constantly pushed back (“ever repulsed”) but always return (“ever rushing on”), as though they have an energy that can’t be overcome (“a life that will not know defeat”).")

P, R, F1 = score([generated_cot], [reference_cot], lang="en", model_type="bert-base-uncased")
print(f"BERTScore F1: {F1.item():.3f}")

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
scores = scorer.score(reference_cot, generated_cot)
print(f"ROUGE-1: {scores['rouge1'].fmeasure:.3f}, ROUGE-L: {scores['rougeL'].fmeasure:.3f}")

BERTScore F1: 0.616
ROUGE-1: 0.366, ROUGE-L: 0.241


# RACE-H
# One shot

# Question Type: Summarization type

In [15]:
#adding [END]
torch.manual_seed(42)
def generate_cot_response_test3(prompt, max_new_tokens=150):  
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
    stop_token_ids = tokenizer.encode("[END]", add_special_tokens=False)
    # Explicitly pass attention_mask to avoid warning
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True, #False is not giving the answer; just produces some text relevant not very meaningful even with increased max_new_tokens
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=stop_token_ids   
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

cot_prompt = (
    '''
[Context]:The United Nations says forty million people or so around the world went hungry in 2008, mainly because of higher food prices. Early estimates from the UN Food and Agriculture Organization (FAO) show that 963 million people did not get enough to eat. World food prices have dropped since early 2008. Prices of major crops have decreased by more than half from their height earlier last year. But they remain high compared to earlier years. But FAO official Hafez Ghanem says lower prices have failed to end the food crisis in many poor countries. "For millions in developing countries," he says, "getting enough food every day to live an active and healthy life is a distant dream." The FAO says food shortage is a threat to people's health. Today, two-thirds of the world's _ people live in just a few countries. These are India, the Democratic Republic of Congo, Bangladesh, Indonesia, Pakistan, Ethiopia and so on. A report on food insecurity warns that the current economic crisis could send even more people into hunger and poverty. In sub-Saharan Africa, the percentage of the people who continually go hungry fell from 34% in 1997 to 30% in 2008. But the FAO says Ghana is the only country that has reached two sets of hunger reduction targets. These were set by the 1996 World Food Summit and the Millennium Development Goals. The main reason is the growth in agricultural production in Ghana. The FAO says some countries in Southeast Asia like Thailand and Vietnam have made progress in hunger reduction goals. But South Asia and Central Asia haven't, and North Korea is still in hot water.
[Question]:What is the best title of this passage?
[Options]:
A) The food production of the world
B) The hunger reduction target of the FAO
C) The food shortage around the world
D) The solution to the global food shortage
Step-by-step reasoning:
Choice C is correct because the passage discusses global hunger statistics, the impact of food prices, and specific regions affected by food shortages, providing a broad overview of the crisis rather than focusing solely on production, targets, or solutions.[END]

[Current Problem]
[Context]: George Bernard Shaw and Winston Churchill disliked each other. It is said that the playwright  once sent Churchill two tickets for the opening night of one of his plays, together with a card, which said, "Bring a friend (if you have one)." 
Churchill, however, returned the tickets with a note, which said, "I shall be busy that evening. Please send me two tickets for the second night (if there is one)." 
There is no record of whether Shaw ever sent the tickets.
[Question]: What was Shaw trying to say to Churchill on his card?
[Options]:
A) Churchill should not go to the play alone.
B) Churchill should not bring too many people.
C) Churchill may have to waste the two tickets.
D) Churchill did not have any friend.
Step-by-step reasoning:
'''
)

# Test it
#print(cot_prompt)

# Diagnostic (optional)
inputs = tokenizer(cot_prompt, return_tensors="pt", padding=True)
print(f"Input length: {inputs['input_ids'].shape[1]} tokens")


# Run and display
print("Generating reasoning...\n")
response = generate_cot_response_test3(cot_prompt)
#print("Full Response:\n", response)

#working - print sboth example and current problem with one reasoning: THIS PRINTS THE ENTIRE RESPONSE
print("RESPONSE:\n", response)

Input length: 629 tokens
Generating reasoning...

RESPONSE:
 
[Context]:The United Nations says forty million people or so around the world went hungry in 2008, mainly because of higher food prices. Early estimates from the UN Food and Agriculture Organization (FAO) show that 963 million people did not get enough to eat. World food prices have dropped since early 2008. Prices of major crops have decreased by more than half from their height earlier last year. But they remain high compared to earlier years. But FAO official Hafez Ghanem says lower prices have failed to end the food crisis in many poor countries. "For millions in developing countries," he says, "getting enough food every day to live an active and healthy life is a distant dream." The FAO says food shortage is a threat to people's health. Today, two-thirds of the world's _ people live in just a few countries. These are India, the Democratic Republic of Congo, Bangladesh, Indonesia, Pakistan, Ethiopia and so on. A report o

In [16]:
from bert_score import score
from rouge_score import rouge_scorer


generated_cot = ('''D is correct. Shaw was trying to tell Churchill that he should not waste the tickets, because he would not be able to see the play at all if he did so. He was also trying to warn Churchill that if he brought too many of his friends, he would be wasting the tickets as well. [''' )
reference_cot = ('''Choice D is correct because the option "Churchill did not have any friend" directly suggests that his lack of companionship is the key issue in the scenario, implying he has no one to share an event or tickets with, unlike the other options which assume he has companions or tickets to manage.''')

P, R, F1 = score([generated_cot], [reference_cot], lang="en", model_type="bert-base-uncased")
print(f"BERTScore F1: {F1.item():.3f}")

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
scores = scorer.score(reference_cot, generated_cot)
print(f"ROUGE-1: {scores['rouge1'].fmeasure:.3f}, ROUGE-L: {scores['rougeL'].fmeasure:.3f}")

BERTScore F1: 0.513
ROUGE-1: 0.370, ROUGE-L: 0.204


# Question type: Fill in the blank


In [17]:
#adding [END]
torch.manual_seed(42)
def generate_cot_response_test3(prompt, max_new_tokens=150):  
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
    stop_token_ids = tokenizer.encode("[END]", add_special_tokens=False)
    # Explicitly pass attention_mask to avoid warning
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True, #False is not giving the answer; just produces some text relevant not very meaningful even with increased max_new_tokens
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=stop_token_ids   
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response


# Inline CoT prompt with one-shot example

cot_prompt =  ('''
[Context]:The South Pole is a particular place on the earth.When people stand at the top of it looking around,they will find all directions face to north.No matter how they make their first step in which direction,they will always walk towards the north.That's to say,only north and south exist there,neither east nor west exists. At the top of the South Pole,any clock or watch keeps good time because all meridians  join there.All kinds of local time are completely suitable there.It is even difficult to distinguish  New Year's Eve from New Year's Day. The explorers and scientists from different countries always fix the time according to their own.Therefore the time by their watches was different when the people with different nationality gathered there. The Winter Solstice  is an important festival at the South Pole.It is always on June 21 or 22.This day is called Midwinter Festival,on which the daytime is the shortest in a year and the night the longest.All people at the South Pole extend greetings to each other and present gifts to each other.They usually give all kinds of celebrations.From that day on,the daytime will get longer day by day."	
[Question]:On the Winter Solstice,  _   is the shortest in a year.	
[Options]:
A) the night
B) the noon	
C) the morning	
D) the daytime	
Step-by-step reasoning\n:
Choice D is correct because the text explicitly states, "the daytime is the shortest in a year and the night the longest" on the Winter Solstice, directly identifying daytime as the shortest period on that day.

[Current Problem]
[Context]:Boxing was long viewed sickly. Generally forbidden by law in earlier days, the fighting was usually done with bare fists, and matches often lasted forty or fifty rounds. In 1882 John L. Sullivan, a fighter of great power, won the world heavyweight championship from Paddy Ryan in a bare fisted battle marked by hitting, scratching, and biting without any rule. Five years later, while fighting Patsy Cardiff at Minneapolis, Sullivan broke his right arm in the third round, but he continued fighting to the sixth round and won. In 1889, Sullivan defeated Jade Kilrain with his bare fists in another championship fight, winning twenty thousand dollars and a diamond prize medal. His admirers talked then of running him for the next governor, but he traveled to Australia for a boxing tour instead, coming back only to lose his title in a twenty-one-round match with a young Californian named James J. Corbett.""Gentleman James"" victory in this match marked a turning point, for it showed scientific boxing was over strength. But Corbett's title ended in 1897, when another boxer, Bob Fitzsimmons, in less than three seconds, achieved his feats and then Fitzsimmons knocked out an Irishman, won the heavyweight championship of the world, and invented the terrible "solar plexus punch."	
[Question]:Sullivan was so popular that his admirers   _  .
[Options]:
A) encouraged him to be a governor	
B) raised twenty thousand dollars for him	
C) advised him to take boxing tour of Australia	
D) refused to believe he could be defeated	
Step-by-step reasoning\n:
''')

# Test it
#print(cot_prompt)

# Diagnostic (optional)
inputs = tokenizer(cot_prompt, return_tensors="pt", padding=True)
print(f"Input length: {inputs['input_ids'].shape[1]} tokens")


# Run and display
print("Generating reasoning...\n")
response = generate_cot_response_test3(cot_prompt)
#print("Full Response:\n", response)

#working - print sboth example and current problem with one reasoning: THIS PRINTS THE ENTIRE RESPONSE
print("RESPONSE:\n", response)


Input length: 702 tokens
Generating reasoning...

RESPONSE:
 
[Context]:The South Pole is a particular place on the earth.When people stand at the top of it looking around,they will find all directions face to north.No matter how they make their first step in which direction,they will always walk towards the north.That's to say,only north and south exist there,neither east nor west exists. At the top of the South Pole,any clock or watch keeps good time because all meridians  join there.All kinds of local time are completely suitable there.It is even difficult to distinguish  New Year's Eve from New Year's Day. The explorers and scientists from different countries always fix the time according to their own.Therefore the time by their watches was different when the people with different nationality gathered there. The Winter Solstice  is an important festival at the South Pole.It is always on June 21 or 22.This day is called Midwinter Festival,on which the daytime is the shortest in a ye

In [18]:
from bert_score import score
from rouge_score import rouge_scorer


generated_cot = ('''Option A is correct, because Sullivan's popularity was such that he was able to be elected governor of New York in 1884.[''' )
reference_cot = ('''Choice A is correct because the text notes that after Sullivan’s victory, "His admirers talked then of running him for the next governor," indicating they encouraged him to pursue a political role rather than raising money, advising travel, or denying his defeat.''')

P, R, F1 = score([generated_cot], [reference_cot], lang="en", model_type="bert-base-uncased")
print(f"BERTScore F1: {F1.item():.3f}")

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
scores = scorer.score(reference_cot, generated_cot)
print(f"ROUGE-1: {scores['rouge1'].fmeasure:.3f}, ROUGE-L: {scores['rougeL'].fmeasure:.3f}")

BERTScore F1: 0.571
ROUGE-1: 0.303, ROUGE-L: 0.212
